# Binary Classification for Sensor Anomaly Detection

This notebook builds a **production-quality end-to-end pipeline** for anomaly detection (`target=1`) using tabular sensor data.

## What this notebook covers
1. Data loading and schema checks  
2. Exploratory data analysis (EDA)  
3. Feature engineering from datetime and sensor readings  
4. Preprocessing and imbalance handling  
5. Multi-model training and comparison  
6. Hyperparameter tuning optimized for **F1 score**  
7. Evaluation (confusion matrix, CV, ROC)  
8. Feature importance  
9. Final predictions and `submission.parquet` export  
10. Bonus: optional Neural Network + learning curves

> The code is written defensively to handle missing optional dependencies (XGBoost, LightGBM, imbalanced-learn, TensorFlow).

In [ ]:
# ==============================
# 0. Imports and Global Settings
# ==============================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV,
    learning_curve,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    RocCurveDisplay,
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

# Optional packages
HAS_IMBLEARN = True
HAS_XGBOOST = True
HAS_LIGHTGBM = True
HAS_TF = True

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
except Exception:
    HAS_IMBLEARN = False

try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMClassifier
except Exception:
    HAS_LIGHTGBM = False

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping
except Exception:
    HAS_TF = False

print("Optional dependency availability:")
print(f"- imbalanced-learn: {HAS_IMBLEARN}")
print(f"- xgboost:          {HAS_XGBOOST}")
print(f"- lightgbm:         {HAS_LIGHTGBM}")
print(f"- tensorflow:       {HAS_TF}")

## 1) Data Loading
Expected files:
- `train.parquet`
- `test.parquet`
- `sample_submission.parquet` (for final output format)

In [ ]:
# =================
# 1. Data Loading
# =================

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.parquet"

assert TRAIN_PATH.exists(), f"Missing file: {TRAIN_PATH.resolve()}"
assert TEST_PATH.exists(), f"Missing file: {TEST_PATH.resolve()}"

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

display(train_df.head())
display(test_df.head())

print("\nTrain info:")
train_df.info()
print("\nTest info:")
test_df.info()

print("\nTrain summary statistics:")
display(train_df.describe(include="all", datetime_is_numeric=True).T)
print("\nTest summary statistics:")
display(test_df.describe(include="all", datetime_is_numeric=True).T)

## 2) Exploratory Data Analysis (EDA)

In [ ]:
# =============================
# 2. EDA: Missing & Imbalance
# =============================

def missing_summary(df, name="dataset"):
    miss = df.isna().sum().sort_values(ascending=False)
    miss_pct = (miss / len(df) * 100).round(2)
    out = pd.DataFrame({"missing_count": miss, "missing_pct": miss_pct})
    out = out[out["missing_count"] > 0]
    print(f"[{name}] Missing columns: {len(out)}")
    display(out if not out.empty else pd.DataFrame({"message": ["No missing values found"]}))

missing_summary(train_df, "train")
missing_summary(test_df, "test")

if "target" in train_df.columns:
    class_counts = train_df["target"].value_counts().sort_index()
    class_pct = (class_counts / class_counts.sum() * 100).round(2)
    print("Class distribution:")
    display(pd.DataFrame({"count": class_counts, "pct": class_pct}))

    plt.figure(figsize=(6, 4))
    sns.countplot(data=train_df, x="target")
    plt.title("Target Class Distribution")
    plt.show()

In [ ]:
# ========================================
# 2. EDA: Feature Distributions & Outliers
# ========================================

sensor_cols = [c for c in ["X1", "X2", "X3", "X4", "X5"] if c in train_df.columns]

train_df[sensor_cols].hist(bins=40, figsize=(14, 8), layout=(2, 3))
plt.suptitle("Sensor Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
sns.boxplot(data=train_df[sensor_cols], orient="h")
plt.title("Sensor Boxplots (Outlier Detection)")
plt.show()

outlier_report = {}
for col in sensor_cols:
    q1 = train_df[col].quantile(0.25)
    q3 = train_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    frac = ((train_df[col] < lower) | (train_df[col] > upper)).mean()
    outlier_report[col] = frac

outlier_df = pd.DataFrame.from_dict(outlier_report, orient="index", columns=["outlier_fraction"]).sort_values("outlier_fraction", ascending=False)
print("IQR outlier fraction by feature:")
display(outlier_df)

In [ ]:
# ============================
# 2. EDA: Correlation Heatmap
# ============================

corr_cols = sensor_cols + (["target"] if "target" in train_df.columns else [])
corr = train_df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap")
plt.show()

## 3) Feature Engineering

In [ ]:
# =======================
# 3. Feature Engineering
# =======================

def add_datetime_features(df, date_col="Date"):
    df = df.copy()
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df["hour"] = df[date_col].dt.hour
        df["day"] = df[date_col].dt.day
        df["month"] = df[date_col].dt.month
        df["weekday"] = df[date_col].dt.weekday
    return df


def add_sensor_aggregates(df, sensors):
    df = df.copy()
    if sensors:
        df["sensor_mean"] = df[sensors].mean(axis=1)
        df["sensor_std"] = df[sensors].std(axis=1)
        df["sensor_min"] = df[sensors].min(axis=1)
        df["sensor_max"] = df[sensors].max(axis=1)
    return df


def add_pairwise_differences(df, sensors):
    df = df.copy()
    for i in range(len(sensors)):
        for j in range(i + 1, len(sensors)):
            c1, c2 = sensors[i], sensors[j]
            df[f"{c1}_minus_{c2}"] = df[c1] - df[c2]
    return df


def engineer_features(df, sensors, date_col="Date"):
    df = add_datetime_features(df, date_col=date_col)
    df = add_sensor_aggregates(df, sensors)
    df = add_pairwise_differences(df, sensors)
    return df

train_fe = engineer_features(train_df, sensor_cols, date_col="Date")
test_fe = engineer_features(test_df, sensor_cols, date_col="Date")

print("Feature-engineered train shape:", train_fe.shape)
print("Feature-engineered test shape :", test_fe.shape)

In [ ]:
# ============================================
# 3. Feature Engineering: low-variance handling
# ============================================

ID_COL = "ID" if "ID" in test_fe.columns else None
TARGET_COL = "target"
DATE_COL = "Date" if "Date" in train_fe.columns else None

drop_cols = [c for c in [TARGET_COL, ID_COL, DATE_COL] if c in train_fe.columns]
feature_cols = [c for c in train_fe.columns if c not in drop_cols]

numeric_feature_cols = train_fe[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

low_var_threshold = 1e-8
variances = train_fe[numeric_feature_cols].var(numeric_only=True)
low_var_cols = variances[variances <= low_var_threshold].index.tolist()

selected_features = [c for c in numeric_feature_cols if c not in low_var_cols]

print(f"Numeric candidate features: {len(numeric_feature_cols)}")
print(f"Dropped low-variance features: {len(low_var_cols)}")
if low_var_cols:
    print("Low-variance columns:", low_var_cols)
print(f"Selected features: {len(selected_features)}")

## 4) Data Preprocessing

In [ ]:
# =====================
# 4. Data Preprocessing
# =====================

X = train_fe[selected_features].copy()
y = train_fe[TARGET_COL].copy().astype(int)

X_test_final = test_fe[selected_features].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train split:", X_train.shape, y_train.shape)
print("Valid split:", X_valid.shape, y_valid.shape)

num_pipe_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

num_pipe_noscale = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

## 5) Model Building (Classical + Advanced)

In [ ]:
# =====================================
# 5. Candidate Models and Pipelines
# =====================================

models = {}

models["log_reg"] = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
models["knn"] = KNeighborsClassifier()
models["svm"] = SVC(
    probability=True,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

models["decision_tree"] = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
models["random_forest"] = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

if HAS_XGBOOST:
    models["xgboost"] = XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        n_jobs=-1,
    )

if HAS_LIGHTGBM:
    models["lightgbm"] = LGBMClassifier(
        random_state=RANDOM_STATE,
        n_estimators=300,
        learning_rate=0.05,
        class_weight="balanced",
        n_jobs=-1,
    )

scaled_models = {"log_reg", "knn", "svm"}


def build_pipeline(estimator, use_scaling=True, use_smote=False):
    preproc = ColumnTransformer(
        transformers=[("num", num_pipe_scaled if use_scaling else num_pipe_noscale, selected_features)],
        remainder="drop",
    )

    if use_smote and HAS_IMBLEARN:
        return ImbPipeline([
            ("preprocess", preproc),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("model", estimator),
        ])

    return Pipeline([
        ("preprocess", preproc),
        ("model", estimator),
    ])

In [ ]:
# ============================================
# 5-8. Baseline Comparison + Model Selection
# ============================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_rows = []
trained_pipelines = {}

for name, model in models.items():
    use_scaling = name in scaled_models
    use_smote = (name in {"log_reg", "knn", "svm"}) and HAS_IMBLEARN

    pipe = build_pipeline(model, use_scaling=use_scaling, use_smote=use_smote)

    f1_cv = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    acc_cv = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_valid)

    row = {
        "model": name,
        "cv_f1_mean": f1_cv.mean(),
        "cv_f1_std": f1_cv.std(),
        "cv_acc_mean": acc_cv.mean(),
        "valid_f1": f1_score(y_valid, y_pred),
        "valid_precision": precision_score(y_valid, y_pred, zero_division=0),
        "valid_recall": recall_score(y_valid, y_pred, zero_division=0),
        "valid_accuracy": accuracy_score(y_valid, y_pred),
    }
    baseline_rows.append(row)
    trained_pipelines[name] = pipe

results_df = pd.DataFrame(baseline_rows).sort_values(
    by=["valid_f1", "cv_f1_mean"], ascending=False
).reset_index(drop=True)

print("Model comparison (sorted by validation F1):")
display(results_df)

best_baseline_name = results_df.iloc[0]["model"]
print(f"Best baseline model candidate: {best_baseline_name}")

## 6) Hyperparameter Tuning (F1-focused)

In [ ]:
# ========================================
# 6. Hyperparameter Tuning (Random Search)
# ========================================

preferred_order = ["lightgbm", "xgboost", "random_forest", "svm", "log_reg"]
chosen_to_tune = next((m for m in preferred_order if m in models), best_baseline_name)
print("Model selected for tuning:", chosen_to_tune)

base_estimator = models[chosen_to_tune]
use_scaling = chosen_to_tune in scaled_models
use_smote = (chosen_to_tune in {"log_reg", "knn", "svm"}) and HAS_IMBLEARN

tune_pipe = build_pipeline(base_estimator, use_scaling=use_scaling, use_smote=use_smote)

param_distributions = {}
if chosen_to_tune == "random_forest":
    param_distributions = {
        "model__n_estimators": [200, 300, 500, 700],
        "model__max_depth": [None, 4, 6, 10, 14],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2", None],
    }
elif chosen_to_tune == "xgboost":
    param_distributions = {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [3, 5, 7, 9],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 1.0],
    }
elif chosen_to_tune == "lightgbm":
    param_distributions = {
        "model__n_estimators": [200, 300, 500, 700],
        "model__num_leaves": [15, 31, 63, 127],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__max_depth": [-1, 4, 6, 10],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 1.0],
    }
elif chosen_to_tune == "svm":
    param_distributions = {
        "model__C": [0.1, 1, 3, 10, 30],
        "model__gamma": ["scale", "auto", 0.1, 0.01, 0.001],
        "model__kernel": ["rbf", "poly"],
    }
else:
    param_distributions = {
        "model__C": np.logspace(-3, 2, 10),
        "model__penalty": ["l2"],
    }

search = RandomizedSearchCV(
    estimator=tune_pipe,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print("Best params:", search.best_params_)
print("Best CV F1:", search.best_score_)

## 7) Evaluation

In [ ]:
# ========================
# 7. Final Model Evaluation
# ========================

y_valid_pred = best_model.predict(X_valid)

if hasattr(best_model, "predict_proba"):
    y_valid_score = best_model.predict_proba(X_valid)[:, 1]
elif hasattr(best_model, "decision_function"):
    y_valid_score = best_model.decision_function(X_valid)
else:
    y_valid_score = None

metrics = {
    "accuracy": accuracy_score(y_valid, y_valid_pred),
    "precision": precision_score(y_valid, y_valid_pred, zero_division=0),
    "recall": recall_score(y_valid, y_valid_pred, zero_division=0),
    "f1": f1_score(y_valid, y_valid_pred),
}

print("Validation metrics:")
for k, v in metrics.items():
    print(f"- {k:10s}: {v:.5f}")

print("\nClassification report:")
print(classification_report(y_valid, y_valid_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(y_valid, y_valid_pred, cmap="Blues")
plt.title("Confusion Matrix (Validation)")
plt.show()

if y_valid_score is not None:
    auc = roc_auc_score(y_valid, y_valid_score)
    print(f"ROC-AUC: {auc:.5f}")
    RocCurveDisplay.from_predictions(y_valid, y_valid_score)
    plt.title("ROC Curve")
    plt.show()

In [ ]:
# =========================
# 7 (Bonus). Learning Curves
# =========================

train_sizes, train_scores, valid_scores = learning_curve(
    estimator=best_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 6),
)

train_mean = train_scores.mean(axis=1)
valid_mean = valid_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker="o", label="Train F1")
plt.plot(train_sizes, valid_mean, marker="o", label="CV F1")
plt.xlabel("Training samples")
plt.ylabel("F1 score")
plt.title("Learning Curve")
plt.legend()
plt.show()

## 8) Feature Importance

In [ ]:
# ======================
# 8. Feature Importance
# ======================

pre = best_model.named_steps["preprocess"]
feature_names_out = pre.get_feature_names_out()

clf = best_model.named_steps["model"]
importances = None

if hasattr(clf, "feature_importances_"):
    importances = clf.feature_importances_
elif hasattr(clf, "coef_"):
    coef = clf.coef_
    importances = np.abs(coef[0] if coef.ndim > 1 else coef)

if importances is not None:
    imp_df = pd.DataFrame({
        "feature": feature_names_out,
        "importance": importances,
    }).sort_values("importance", ascending=False)

    display(imp_df.head(20))

    plt.figure(figsize=(10, 6))
    sns.barplot(data=imp_df.head(20), x="importance", y="feature", orient="h")
    plt.title("Top 20 Feature Importances")
    plt.tight_layout()
    plt.show()
else:
    print("Selected model does not expose direct feature importance.")

## 9) Final Training + Test Prediction + Submission

In [ ]:
# ==========================================
# 9. Retrain on full train data and infer test
# ==========================================

best_model.fit(X, y)

test_pred = best_model.predict(X_test_final).astype(int)

if SAMPLE_SUB_PATH.exists():
    submission = pd.read_parquet(SAMPLE_SUB_PATH)
    submission["target"] = test_pred

    if "ID" in submission.columns and "ID" in test_fe.columns:
        submission["ID"] = test_fe["ID"].values
else:
    if "ID" in test_fe.columns:
        submission = pd.DataFrame({"ID": test_fe["ID"], "target": test_pred})
    else:
        submission = pd.DataFrame({"target": test_pred})

submission_path = DATA_DIR / "submission.parquet"
submission.to_parquet(submission_path, index=False)

print(f"Saved submission file to: {submission_path.resolve()}")
display(submission.head())

## 10) Optional Bonus: Neural Network (TensorFlow)

This section is optional and runs only when TensorFlow is installed.

In [ ]:
# ======================================
# 10. Bonus Neural Network (Optional)
# ======================================

if HAS_TF:
    pre_nn = ColumnTransformer(
        transformers=[("num", num_pipe_scaled, selected_features)],
        remainder="drop",
    )

    X_train_nn = pre_nn.fit_transform(X_train)
    X_valid_nn = pre_nn.transform(X_valid)

    model_nn = Sequential([
        Dense(128, activation="relu", input_shape=(X_train_nn.shape[1],)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(1, activation="sigmoid"),
    ])

    model_nn.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
    )

    callbacks = [EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]

    history = model_nn.fit(
        X_train_nn,
        y_train,
        validation_data=(X_valid_nn, y_valid),
        epochs=100,
        batch_size=256,
        callbacks=callbacks,
        verbose=0,
    )

    y_nn_prob = model_nn.predict(X_valid_nn, verbose=0).ravel()
    y_nn_pred = (y_nn_prob >= 0.5).astype(int)

    print("Neural Network Validation Metrics:")
    print("F1:", f1_score(y_valid, y_nn_pred))
    print("Precision:", precision_score(y_valid, y_nn_pred, zero_division=0))
    print("Recall:", recall_score(y_valid, y_nn_pred, zero_division=0))

    plt.figure(figsize=(8, 4))
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title("Neural Network Training History")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()
else:
    print("TensorFlow not available; skipping neural network section.")

## 11) Model Selection Rationale (Template)

After running this notebook, choose the final model using:
- Highest **Validation F1** and **Cross-validated F1 mean**
- Stable precision/recall tradeoff (for anomaly detection, recall is often critical)
- Consistency between CV and holdout results (low overfitting gap)
- Operational constraints (inference speed, interpretability)

In many tabular anomaly tasks, tree-boosting models (LightGBM/XGBoost) tend to perform best; however, this notebook selects empirically from your data rather than assuming.